# Week 4 Discussion: Debugging Code

## 1. Load data

In [1]:
# Load packages
import io
import pandas as pd

# Define sample dataset
CSV = """site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
SB-01,2024-01-15,12.4,"1,240",0.82,good
SB-01,2024-02-15,13.1,"1,180",0.79,good
SB-01,2024-03-15,15.8,"2,050",1.21,good
SB-01,2024-04-15,18.2,"1,620",0.94,good
sb-01,2024-05-15,21.5,"980",0.66,good
SB-01 ,2024-06-15,-999,"540",0.51,suspect
SB-02,2024-01-15,11.8,"3,410",1.88,good
SB-02,2024-02-15,12.3,"3,290",1.92,good
SB-02,2024-03-15,14.9,"4,870",2.41,good
SB-02,2024-04-15,17.6,"3,950",2.09,estimated
SB-02,2024-05-15,20.9,"2,240",1.44,good
sb-02,2024-06-15,-999,"1,130",1.02,suspect
SB-03,2024-01-15,10.2,"620",<0.05,good
SB-03,2024-02-15,10.9,"590",<0.05,good
SB-03,2024-03-15,13.4,"1,040",0.33,good
SB-03,2024-04-15,16.1,"870",0.28,good
SB-03 ,2024-05-15,19.3,"410",<0.05,good
SB-03,2024-06-15,-999,"180",<0.05,suspect
"""

# Load in sample data
stream = pd.read_csv(io.StringIO(CSV))

In [2]:
# Print dataframe
stream

,site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
0,SB-01,2024-01-15,12.4,"1,240",0.82,good
1,SB-01,2024-02-15,13.1,"1,180",0.79,good
2,SB-01,2024-03-15,15.8,"2,050",1.21,good
3,SB-01,2024-04-15,18.2,"1,620",0.94,good
4,sb-01,2024-05-15,21.5,980,0.66,good
5,SB-01,2024-06-15,-999.0,540,0.51,suspect
6,SB-02,2024-01-15,11.8,"3,410",1.88,good
7,SB-02,2024-02-15,12.3,"3,290",1.92,good
8,SB-02,2024-03-15,14.9,"4,870",2.41,good
9,SB-02,2024-04-15,17.6,"3,950",2.09,estimated


- How many monitoring sites are in this dataset?

    There are three monitoring sites.

- What is a plausible range for stream water temperature in degrees Celsius?

    10 to 22 degrees C
  
- Roughly what mean water temperature do you expect across all measurements?

    Mean is approximately 16 degrees C

## 2. A worked example

In [3]:
# First attempt to convert stream discharge to cubic meters per second
stream['discharge_cfs'] * 0.0283168

TypeError: can't multiply sequence by non-int of type 'float'

In [4]:
# Look at the data
stream.dtypes

site_id           object
date              object
water_temp_c     float64
discharge_cfs     object
nitrate_mg_l      object
qc_flag           object
dtype: object

In [5]:
# Isolate 
stream['discharge_cfs'].iloc[0] * 0.0283168   # same TypeError, one value
stream['discharge_cfs'].iloc[0]              # '1,240'

TypeError: can't multiply sequence by non-int of type 'float'

In [6]:
# Form hypothesis and test it
# Hypothesis: thousands separator (,) is forcing the values in discharge_cfs to text
stream['discharge_cfs'].unique()

array(['1,240', '1,180', '2,050', '1,620', '980', '540', '3,410', '3,290',
       '4,870', '3,950', '2,240', '1,130', '620', '590', '1,040', '870',
       '410', '180'], dtype=object)

In [7]:
# Fix and verify 
discharge_cms = (stream['discharge_cfs']
                    .str.replace(',', '', regex=False) # Remove commas
                    .astype(float)) * 0.0283168 # Convert to float data type and multiply

discharge_cms.head(3) # Verify

0    35.112832
1    33.413824
2    58.049440
Name: discharge_cfs, dtype: float64

In [8]:
# Consider adding a guard
assert pd.api.types.is_numeric_dtype(discharge_cms), 'Discharge failed to convert to numeric'

If the multiplication still failed after completing the debugging steps, the next step is to build a **minimal reproducible example** and ask for help. 

In [9]:
# Create minimal reproducible example
import pandas as pd
df = pd.DataFrame({'discharge_cfs': ['1,240', '980']})
df['discharge_cfs'] * 0.0283168     # TypeError: can't multiply sequence by non-int

TypeError: can't multiply sequence by non-int of type 'float'

Finally, create a log to record what happened.

| What I expected | What happened | My hypothesis | What I changed | Result |
| --------------- | ------------- | ------------- | -------------- | ------ |
| Discharge in m³/s, around 35 | `TypeError: can't multiply sequence by non-int` | `discharge_cfs` is text because of thousands separators |	`.str.replace(',', '')` then `.astype(float)` |	Works, 35.11 m³/s, guard added |

## 3. On your own
Four cells below are broken. No steps are given for any of them. For each one, predict what will happen before you run it, then work the full protocol, fix it, and add a row to your log.

A. The intention is to strip blanks from the site_id codes.

In [10]:
stream['site_id'].strip()

AttributeError: 'Series' object has no attribute 'strip'

In [12]:
# Step 1: Look at the data
print(stream['site_id'].dtype)

object


In [16]:
# Step 2: Isolate
print(stream['site_id'].iloc[0].strip())
print(type(stream['site_id'].iloc[0]))

SB-01
<class 'str'>


After isolation, the first value in site_id was returned. Using .strip() after .iloc[] turned the value into a string. The smalled segment of code that still causes the error is the original`stream['site_id'].strip()`

In [17]:
# Step 3. Form a hypothesis and test it
# Our hypothesis is that .str needs to precede .strip() for Series objects
stream['site_id'].str.strip()

0     SB-01
1     SB-01
2     SB-01
3     SB-01
4     sb-01
5     SB-01
6     SB-02
7     SB-02
8     SB-02
9     SB-02
10    SB-02
11    sb-02
12    SB-03
13    SB-03
14    SB-03
15    SB-03
16    SB-03
17    SB-03
Name: site_id, dtype: object

The above step fixed the error and verified it. 

| Error | What I expected | What happened | My hypothesis | What I changed | Result |
|-------| --------------- | ------------- | ------------- | -------------- | ------ |
| Example | Discharge in m³/s, around 35 | `TypeError: can't multiply sequence by non-int` | `discharge_cfs` is text because of thousands separators |	`.str.replace(',', '')` then `.astype(float)` |	Works, 35.11 m³/s, guard added |
| A | Remove spaces from site id | `AttributeError: 'Series' object has no attribute 'strip'` | .str needs to precede .strip() for Series objects |	Used `stream['site_id'].str.strip()` |	Works |